In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "sentence-transformers/natural-questions",
    "pair",
    split="train"
)

device = torch.device("cuda")
model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)
model.to(device)

README.md:   0%|          | 0.00/2.28k [00:00<?, ?B/s]

pair/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 44.0MB            

pair/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/100231 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [4]:
dataset


Dataset({
    features: ['query', 'answer'],
    num_rows: 100231
})

In [2]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list, padding=True, truncation=True, return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

In [5]:
embeddings_dataset = dataset.map(lambda x: {"embeddings": get_embeddings(x["answer"]).detach().cpu().numpy()[0]})

Map:   0%|          | 0/100231 [00:00<?, ? examples/s]

In [11]:

import faiss

In [ ]:
%pip install faiss-cpu

In [13]:
embeddings_dataset.add_faiss_index(column="embeddings")

ImportError: You must install Faiss to use FaissIndex. To do so you can run `conda install -c pytorch faiss-cpu` or `conda install -c pytorch faiss-gpu`. A community supported package is also available on pypi: `pip install faiss-cpu` or `pip install faiss-gpu`. Note that pip may not have the latest version of FAISS, and thus, some of the latest features and bug fixes may not be available.

In [14]:
import faiss
import datasets.search

datasets.search._has_faiss = True

embeddings_dataset.add_faiss_index(column="embeddings")

  0%|          | 0/101 [00:00<?, ?it/s]

Dataset({
    features: ['query', 'answer', 'embeddings'],
    num_rows: 100231
})

In [18]:
embeddings_dataset[0]['embeddings']

[-0.37552493810653687,
 -0.35448336601257324,
 -0.07888665795326233,
 0.15109457075595856,
 0.17561985552310944,
 -0.15682975947856903,
 -0.5072841048240662,
 0.008823756128549576,
 -0.37989485263824463,
 -0.029966769739985466,
 0.23192782700061798,
 -0.21263796091079712,
 0.019411452114582062,
 0.03353070467710495,
 0.3422435224056244,
 -0.09191901981830597,
 -0.23274938762187958,
 -0.09553205221891403,
 0.5223147869110107,
 -0.11889146268367767,
 -0.1560189574956894,
 -0.15035676956176758,
 0.1540449559688568,
 0.08463163673877716,
 0.04877547547221184,
 0.31988897919654846,
 -0.17085252702236176,
 0.047196779400110245,
 -0.31743529438972473,
 -0.584308922290802,
 0.17768362164497375,
 0.18311907351016998,
 -0.02992120012640953,
 0.2701866924762726,
 -0.00011681287287501618,
 -0.23340803384780884,
 -0.03047809563577175,
 -0.2680661380290985,
 -0.14653877913951874,
 0.010359484702348709,
 0.22716258466243744,
 0.2439158409833908,
 0.011165888980031013,
 -0.27317431569099426,
 -0.20951

# Inference

In [19]:
question = "How can I load a dataset offline?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

(1, 768)

In [20]:
scores, samples = embeddings_dataset.get_nearest_examples(
    "embeddings", question_embedding, k=5)



samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)

In [21]:
samples_df

,query,answer,embeddings,scores
4,why you shouldn't use wikipedia for research,"Wikipedia:Academic use However, citation of Wi...","[0.22036714851856232, -0.05654790624976158, -0...",53.833809
3,where would a proxy server be located in a net...,Proxy server A proxy server may reside on the ...,"[0.31574681401252747, -0.21349899470806122, -0...",53.781578
2,to launch a distributed denial of service (ddo...,Denial-of-service attack A wide array of progr...,"[-0.11166858673095703, -0.5228753089904785, -0...",53.394695
1,what does the little u mean in statistics,Standard deviation Let X be a random variable ...,"[-0.27044638991355896, -0.7066569924354553, -0...",53.241951
0,common methods of data collection for job anal...,Data analysis Data is collected from a variety...,"[0.2494194507598877, 0.3156343996524811, -0.18...",51.835861
